# БДЗ 2

In [1]:
import sys
# sys.path.append("src")
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import warnings
warnings.filterwarnings('ignore', message='.*Converting mask without torch.bool.*')

In [2]:
import torch
from torch.utils.data import DataLoader
import torchtext
from pathlib import Path
import sacrebleu
from torch import nn

from src.dataset import TranslationDataset
from src.data_utils import create_token_dataloader, CollateFn
from src.inference import translate_file
from src.train import train, InverseSqrtScheduler
from src.prepare_spm import train_sentencepiece
from src.models.rnn import Seq2Seq
from src.models.transformer import TransformerMT
from src.models.transformer2 import TransformerMT2
from src.models.transformer_best import TransformerBest

In [3]:
print(f"PyTorch версия: {torch.__version__}")
print(f"torchtext версия: {torchtext.__version__}")
print(f"sacrebleu версия: {sacrebleu.__version__}")

PyTorch версия: 2.10.0
torchtext версия: 0.6.0
sacrebleu версия: 2.6.0


In [4]:
# обучение sentencepiece

DATA_DIR = "data"
FORCE_RETRAIN_SPM = False # принудительное пересоздание модели SentencePiece
VOCAB_SIZE = 16000

SPM_MODEL_PATH = Path(DATA_DIR) / "sentencepiece.model"

if not SPM_MODEL_PATH.exists() or FORCE_RETRAIN_SPM:
    print("SentencePiece model not found. Training...")
    train_sentencepiece(
        data_dir=DATA_DIR,
        vocab_size=VOCAB_SIZE,
        model_prefix="unigram"
    )
else:
    print("SentencePiece model already exists.")

SentencePiece model already exists.


In [5]:
# даталоадеры

train_ds = TranslationDataset(
    "train.de-en.de",
    "train.de-en.en",
    "sentencepiece.model",
    DATA_DIR
)

val_ds = TranslationDataset(
    "val.de-en.de",
    "val.de-en.en",
    "sentencepiece.model",
    DATA_DIR
)

BATCH_SIZE = 64
PAD = train_ds.PAD
vocab_size = train_ds.vocab_size + 3

# батч по токенам 
train_loader = create_token_dataloader(train_ds, tokens_per_batch=8000)
val_loader = create_token_dataloader(val_ds, tokens_per_batch=8000)

# батч по текстам
# train_loader = DataLoader(
#     train_ds,
#     batch_size=BATCH_SIZE,
#     shuffle=True,
#     collate_fn=CollateFn(PAD)
# )

# val_loader = DataLoader(
#     val_ds,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     collate_fn=CollateFn(PAD)
# )

In [6]:
# обрезка датасета для быстрого тестирования
DEBUG = True
DEBUG_SAMPLES = 20000

if DEBUG:
    train_ds.src_lines = train_ds.src_lines[:DEBUG_SAMPLES]
    train_ds.tgt_lines = train_ds.tgt_lines[:DEBUG_SAMPLES]
    
    # val_ds.src_lines = val_ds.src_lines[:64]
    # val_ds.tgt_lines = val_ds.tgt_lines[:64]

    print("Running in DEBUG mode")
    print("Train size:", len(train_ds))
    print("Val size:", len(val_ds))

Running in DEBUG mode
Train size: 20000
Val size: 986


In [7]:
# модель

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
# device = torch.device("cpu")


# FIRST TRANSFORMER

# model = TransformerMT(
#     vocab_size=vocab_size,
#     emb_size=512,
#     nhead=8,
#     num_encoder_layers=6,
#     num_decoder_layers=6,
#     dim_feedforward=2048,
#     dropout=0.1,
#     pad_idx=train_ds.PAD,
#     max_len=512
# ).to(device)

# SECOND TRANSFORMER

# model = TransformerMT2(
#     vocab_size=vocab_size,
#     d_model=512,
#     nhead=8,
#     num_encoder_layers=6,
#     num_decoder_layers=6,
#     dim_feedforward=2048,
#     dropout=0.1,
#     pad_idx=PAD,
#     max_len=512,
# ).to(device)

# BEST TRANSFORMER

model = TransformerBest(
    vocab_size=vocab_size,
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048,
    pad_idx=PAD
).to(device)


/Users/andrey/study/dl-1/hw/bhw-2/.venv/lib/python3.13/site-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


In [8]:
# параметры обучения
NUM_EPOCHS = 50


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=2e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)

# scheduler = torch.optim.lr_scheduler.OneCycleLR(
#     optimizer,
#     max_lr=5e-4,
#     epochs=NUM_EPOCHS,
#     steps_per_epoch=len(train_loader),
#     pct_start=0.1,
#     anneal_strategy='cos',
#     div_factor=25,
#     final_div_factor=1000,
# )


scheduler = InverseSqrtScheduler(optimizer, d_model=512, warmup_steps=4000, factor=1.0)

criterion = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.10)

In [ ]:
train_losses, val_losses, train_ppls, val_ppls, val_bleus = train(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader,
    pad_idx=PAD,
    num_epochs=NUM_EPOCHS,
    val_dataset=val_ds,
    max_decoding_len=50,
    bleu_every=5,
    tmp_val_out="outputs/val_predictions.en",
    device=device,
    plot=True,
    inference_batch_size=64,
    no_repeat_ngram_size=3
)

Train 1/50: 0it [00:00, ?it/s]

In [ ]:
# BLEU на валидации

val_src = open("data/val.de-en.de", encoding="utf-8").read().splitlines()
val_ref = open("data/val.de-en.en", encoding="utf-8").read().splitlines()

VAL_OUTPUT_PATH = "outputs/val_predictions.en"

translate_file(
    model=model,
    dataset=val_ds,
    input_lines=val_src,
    device=device,
    output_path=VAL_OUTPUT_PATH,
    max_decoding_len=50,
    no_repeat_ngram_size=3
)

val_pred = open(VAL_OUTPUT_PATH, encoding="utf-8").read().splitlines()

bleu = sacrebleu.corpus_bleu(val_pred, [val_ref], tokenize='none')
print("Validation BLEU:", bleu.score)

Validation BLEU: 0.0013502753613091464


In [ ]:
# примеры вывода

val_pred = open("outputs/val_predictions.en", encoding="utf-8").read().splitlines()
val_ref  = open("data/val.de-en.en", encoding="utf-8").read().splitlines()

print("counts: pred =", len(val_pred), "ref =", len(val_ref))

for i in range(10):
    print(f"\n=== Example {i} ===")
    print("PRED repr:", repr(val_pred[i]))
    print("REF  repr:", repr(val_ref[i]))

counts: pred = 986 ref = 986

=== Example 0 ===
PRED repr: 'atoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratorator nuatoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratorator'
REF  repr: 'when i was 11 , i remember waking up one morning to the sound of joy in my house .'

=== Example 1 ===
PRED repr: 'atoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratoratorator'
REF  repr: 'my father was listening to bbc news on his small , gray radio .'

=== Example 2 ===
PRED repr: 'made made made made carround car carround car carroundator car car car medizin car car schlimm car carround carround schlimmator carator made car caratorator carator car medizinatoratorator carroundator made caratorroundround medizin'
REF  repr: 'there was a big smile on his face which was unusual the

In [ ]:
# прогноз для тестового набора

test_lines = open("data/test1.de-en.de", encoding="utf-8").read().splitlines()

OUTPUT_PATH = "outputs/test1.de-en.en"

translate_file(
    model=model,
    dataset=train_ds,
    input_lines=test_lines,
    device=device,
    output_path=OUTPUT_PATH,
    max_decoding_len=50
)

print(f"Test translations saved to {OUTPUT_PATH}")

Test translations saved to outputs/test1.de-en.en
